# MedCLIPSeg on Google Colab — v0.2

Runs the official [MedCLIPSeg](https://github.com/HealthX-Lab/MedCLIPSeg) code (CVPR 2026) on Colab, saves every result to Google Drive as it is produced, and adds CPU‑only analysis sections.

**What's new in v0.1**
* **Nothing is lost when Colab recycles the VM:** every `evaluate()` immediately writes its summary row, its per‑image CSV and (optionally) its predicted masks + uncertainty maps to Drive. `save_tables()` writes all tables to one Excel workbook + CSVs.
* **Section R** restores what the first session saved (`results_*.csv`, `csv_logs_*.tgz`).
* **Section A – data understanding** and **Section B – claim checks** run **without a GPU**.
* **Tier 2d – controlled prompt edits:** change one prompt attribute at a time (the repo's prompt "designs" change several at once).

| Section | Needs GPU? | What |
|---|---|---|
| 0 Setup, data | no | env, patches, datasets |
| R Restore | no | import session‑1 results into the new format |
| A Data understanding | no | prompt inventory, attribute consistency across prompt designs, prompt ↔ mask agreement |
| B Claim checks | no (slow‑ish on CPU) | normal images per prompt design, Dice‑loss shape, logit and soft‑target diagnostics |
| T1–T2 | **yes** | released‑checkpoint evaluation, prompt robustness, MC sampling |
| T3 | **yes** | training from scratch |

Drive layout (`DRIVE_DIR/results/`): `all_results.csv` · `per_image/<run>.csv` · `maps/<run>.zip` · `train_logs/` · `tables/medclipseg_tables.xlsx` (+ one CSV per table).

## 0. Setup (works on CPU or GPU runtimes)

In [ ]:
import torch, sys, subprocess
HAS_GPU = torch.cuda.is_available()
print("python", sys.version.split()[0], "| torch", torch.__version__, "| GPU:", HAS_GPU)
if HAS_GPU:
    print(subprocess.run("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader", shell=True, capture_output=True, text=True).stdout)
else:
    print("CPU runtime: sections 0, R, A, B work. Tiers T1-T3 need a GPU (Runtime > Change runtime type).")

In [ ]:
# ---- EDIT THESE ----
REPO_URL  = "https://github.com/EhsanSl/MedCLIPSeg.git"
BRANCH    = "main"
USE_DRIVE = True                                    # keep zips + results on Google Drive between sessions
DRIVE_DIR = "/content/drive/MyDrive/MedCLIPSeg_colab"
SEED      = 666                                     # seed used for all released checkpoints
SAVE_MAPS = True                                    # zip predicted masks + uncertainty maps of every evaluation to Drive
KEEP_CKPT = False                                   # copy each trained checkpoint (~1 GB) to Drive after training
# --------------------
import os
REPO = "/content/MedCLIPSeg"
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    os.makedirs(f"{DRIVE_DIR}/zips", exist_ok=True)
if not os.path.isdir(REPO):
    !git clone -b {BRANCH} {REPO_URL} {REPO}
else:
    !git -C {REPO} pull --ff-only
%cd {REPO}
!git log --oneline -1

### Install dependencies
`requirements.txt` pins `torch==2.9.0` and `numpy<2.0`. On Colab we **keep the preinstalled torch/torchvision/numpy** (avoids a ~2 GB download and a forced restart). pip will print an `ERROR ... gradio / diffusers requires huggingface-hub>=1...` line: harmless, MedCLIPSeg never imports those packages.

In [ ]:
!grep -v -E "^(torch|torchvision|numpy)([=<>]|$)" requirements.txt > /tmp/req_colab.txt
!pip install -q -r /tmp/req_colab.txt
import importlib
for m in ["torch", "torchvision", "numpy", "transformers", "monai", "timm", "cv2", "huggingface_hub", "pandas"]:
    print(f"{m:16s}", importlib.import_module(m).__version__)

### Compatibility patches (idempotent)
1. `np.Inf → np.inf` (NumPy 2) — otherwise `utils/eval.py` crashes.
2. DataLoader `num_workers=8` → number of Colab CPUs.
3. **Bug fix:** `train.py --resume` looked for the checkpoint in `trained_models/` instead of `trained_models/seed{N}/`, so resuming after a disconnect silently restarted from epoch 0.

These only touch the Colab copy of the repo; `git diff` shows exactly what changed.

In [ ]:
%%writefile /content/colab_patches.py
"""Small compatibility/robustness patches for running MedCLIPSeg on Colab. Idempotent."""
import os, re, pathlib
def _read(path):
    raw = open(path, newline="").read(); crlf = "\r\n" in raw
    return raw.replace("\r\n", "\n"), crlf
def _write(path, s, crlf):
    open(path, "w", newline="").write(s.replace("\n", "\r\n") if crlf else s)
def patch(path, old, new):
    s, crlf = _read(path)
    if new in s: print(f"[skip] {path}: already patched"); return
    assert old in s, f"pattern not found in {path}: {old[:60]!r}"
    _write(path, s.replace(old, new), crlf); print(f"[ok]   {path}")
# 1) NumPy 2.x removed np.Inf (Colab ships NumPy 2) -> eval.py crashes without this
s, crlf = _read("utils/SurfaceDice.py")
_write("utils/SurfaceDice.py", re.sub(r"np\.Inf\b", "np.inf", s), crlf); print("[ok]   utils/SurfaceDice.py (np.Inf -> np.inf)")
# 2) DataLoader workers: repo hard-codes 8; Colab has 2-12 vCPUs
n = min(8, os.cpu_count() or 2)
patch("train.py", "num_workers=8,\n                            pin_memory=True,)", f"num_workers={n},\n                            pin_memory=True,)")
patch("train.py", "num_workers=8,\n                        pin_memory=True)", f"num_workers={n},\n                        pin_memory=True)")
# 3) --resume looked in the wrong folder (missing seed{N}/), so it silently restarted from epoch 0
patch("train.py",
      '            "trained_models",\n            f"{results_name}_latest.pth")',
      '            "trained_models",\n            f"seed{cfg.seed}",\n            f"{results_name}_latest.pth")')
patch("train.py", "checkpoint = torch.load(resume_path)", "checkpoint = torch.load(resume_path, map_location=cfg.MODEL.DEVICE, weights_only=False)")

In [ ]:
!python /content/colab_patches.py
!git diff --stat

In [ ]:
# Cache the UniMedCLIP backbone weights on Drive so they are downloaded only once
if USE_DRIVE:
    os.makedirs(f"{DRIVE_DIR}/checkpoints", exist_ok=True)
    if not os.path.islink("checkpoints"):
        !rm -rf checkpoints
        os.symlink(f"{DRIVE_DIR}/checkpoints", "checkpoints")
!ls -la checkpoints/ 2>/dev/null | head

### Data
Datasets come from the authors' Hugging Face repo (`TahaKoleilat/MedCLIPSeg`, one zip per dataset; zips are cached on Drive, so later sessions only unzip). The repo already contains the text prompts (`data/<DS>/Prompts_Folder`); the zips add `Train_Folder / Val_Folder / Test_Folder` with `img/` and `label/`.

Groups: **BUS** = BUSI → BUSBRA, BUSUC, BUID, UDIAT · **ENDO** = Kvasir → ColonDB, ClinicDB, CVC300, BKAI · **DERM** = ISIC → UWaterlooSkinCancer · **BRAIN** = BTMRI → BRISC.

In [ ]:
import shutil, zipfile, glob
from huggingface_hub import hf_hub_download

ZIP_DIR = f"{DRIVE_DIR}/zips" if USE_DRIVE else "/content/zips"

def get_dataset(name):
    dst = f"data/{name}"
    if glob.glob(f"{dst}/Test_Folder/img/*"):
        print(f"[have] {name}"); return
    zpath = hf_hub_download(repo_id="TahaKoleilat/MedCLIPSeg", repo_type="dataset",
                            filename=f"{name}.zip", local_dir=ZIP_DIR)
    tmp = f"/content/_unzip/{name}"
    shutil.rmtree(tmp, ignore_errors=True)
    with zipfile.ZipFile(zpath) as z: z.extractall(tmp)
    root = next(d for d, sub, _ in os.walk(tmp) if "Test_Folder" in sub or "Train_Folder" in sub)
    for item in os.listdir(root):
        src, out = os.path.join(root, item), os.path.join(dst, item)
        if item == "Prompts_Folder":            # keep the repo's prompt files, only add missing ones
            os.makedirs(out, exist_ok=True)
            for f in os.listdir(src):
                if not os.path.exists(os.path.join(out, f)): shutil.copy(os.path.join(src, f), out)
        elif os.path.isdir(src): shutil.copytree(src, out, dirs_exist_ok=True)
    shutil.rmtree(tmp, ignore_errors=True)
    print(f"[ok]   {name}")

BUS = ["BUSI", "BUSBRA", "BUSUC", "BUID", "UDIAT"]
for ds in BUS:
    get_dataset(ds)

In [ ]:
# Sanity check: every image listed in the prompt files exists, and count empty ground-truth masks
import pandas as pd, numpy as np, cv2
def check(name):
    rows = []
    for split, xl in [("Train", "Train_text.xlsx"), ("Val", "Val_text.xlsx"), ("Test", "Test_text_original.xlsx")]:
        p = f"data/{name}/Prompts_Folder/{xl}"
        if not os.path.exists(p): continue
        df = pd.read_excel(p)
        folder = f"data/{name}/{split}_Folder"
        missing = sum(not os.path.exists(f"{folder}/img/{i}") for i in df["Image"])
        empty = sum(cv2.imread(f"{folder}/label/{g}", 0).max() == 0 for g in df["Ground Truth"]
                    if os.path.exists(f"{folder}/label/{g}"))
        rows.append(dict(dataset=name, split=split, prompts=len(df), missing_imgs=missing, empty_masks=empty))
    return rows
display(pd.DataFrame(sum([check(d) for d in BUS], [])))

### Helpers: evaluate, persist, compare with the paper
`evaluate()` runs `test.py` + `utils/eval.py`, then **immediately** saves to Drive:
* a summary row → `results/all_results.csv` (reloaded automatically next session),
* the per‑image CSV, enriched with `gt_empty`, `pred_empty`, areas and the prompt text → `results/per_image/<run_id>.csv` (unique name per run, so nothing gets overwritten),
* masks + uncertainty maps → `results/maps/<run_id>.zip` (if `SAVE_MAPS`).

Summary columns: `DSC_nonempty` = DSC over images whose ground truth has a lesion; `normals_pred_empty` = empty‑GT images where the model predicted nothing (each scores DSC = 1); `lesions_missed` = lesion images where the model predicted nothing.

In [ ]:
import re, time, datetime
RES_DIR = f"{DRIVE_DIR}/results" if USE_DRIVE else "/content/results"
PER_IMG_DIR, MAP_DIR, LOG_DIR, TAB_DIR = [f"{RES_DIR}/{d}" for d in ["per_image", "maps", "train_logs", "tables"]]
for d in [RES_DIR, PER_IMG_DIR, MAP_DIR, LOG_DIR, TAB_DIR]: os.makedirs(d, exist_ok=True)
ALL_CSV = f"{RES_DIR}/all_results.csv"
RESULTS = pd.read_csv(ALL_CSV).to_dict("records") if os.path.exists(ALL_CSV) else []
TABLES = {}          # analysis tables collected for save_tables()
print(f"{len(RESULTS)} result rows loaded from {ALL_CSV}")

PAPER = {  # MedCLIPSeg DSC (%) from the paper: Tables S14–S17 (data efficiency) and Table 2 (domain generalisation)
    ("BUSI",10):68.66, ("BUSI",25):77.73, ("BUSI",50):81.48, ("BUSI",100):85.72,
    ("BTMRI",10):79.07, ("BTMRI",25):83.93, ("BTMRI",50):85.93, ("BTMRI",100):88.03,
    ("ISIC",10):90.35, ("ISIC",25):91.00, ("ISIC",50):91.97, ("ISIC",100):92.54,
    ("Kvasir",10):77.21, ("Kvasir",25):84.21, ("Kvasir",50):88.18, ("Kvasir",100):90.15,
    ("Covid19",10):79.73, ("Covid19",25):81.83, ("Covid19",50):82.97, ("Covid19",100):83.41,
    ("BUSBRA",100):75.06, ("BUSUC",100):84.37, ("BUID",100):78.99, ("UDIAT",100):74.64,
    ("ColonDB",100):71.90, ("ClinicDB",100):80.80, ("CVC300",100):80.82, ("BKAI",100):79.15,
    ("BRISC",100):80.92, ("UWaterlooSkinCancer",100):83.53,
}
SUB = "MedCLIPSeg_unimedclip_ViT-B-16_Prompt-{}"
designs = ["original", "missing_location", "underdescriptive", "overdescriptive", "contradictory", "shuffled"]
NORMAL_RE = r"\bno (?:tumou?r|polyp|lesion)|\bnormal\b|unremarkable|healthy"
import matplotlib.pyplot as plt, textwrap

def sh(cmd):
    print("$", cmd)
    p = subprocess.run(cmd, shell=True, text=True, capture_output=True)
    if p.returncode != 0:
        print(p.stdout[-3000:]); print(p.stderr[-6000:]); raise RuntimeError("command failed")
    return p.stdout

def slug(s): return re.sub(r"[^A-Za-z0-9.]+", "-", str(s)).strip("-") or "default"
def run_id(target, source, pct, prompt, seed, opts, out):
    return f"{slug(out.replace('outputs_', ''))}__{source}-{target}__p{pct}__{prompt}__s{seed}__{slug(opts)}"

def prompts_for(target, prompt):
    return pd.read_excel(f"data/{target}/Prompts_Folder/Test_text_{prompt}.xlsx").set_index("Ground Truth")["Description"].astype(str).str.strip()

def enrich(df, target, prompt, seg_dir=None):
    # add ground-truth / prediction facts per image
    lab = f"data/{target}/Test_Folder/label"
    gt = [cv2.imread(f"{lab}/{n}", 0) for n in df.Name]
    df["gt_empty"] = [g.max() == 0 for g in gt]
    df["gt_area"] = [float((g > 127).mean()) for g in gt]
    if seg_dir:
        pr = [cv2.imread(f"{seg_dir}/{n}", 0) for n in df.Name]
        df["pred_empty"] = [p.max() == 0 for p in pr]
        df["pred_area"] = [float((p > 127).mean()) for p in pr]
    else:   # legacy: only known for empty-GT images (DSC==1 <=> nothing predicted)
        df["pred_empty"] = np.where(df.gt_empty, df.DSC == 1, np.nan)
    try: df["prompt_text"] = df.Name.map(prompts_for(target, prompt))
    except FileNotFoundError: df["prompt_text"] = None
    return df

def summarize(df, **meta):
    ge, pe = df.gt_empty.astype(bool), df.pred_empty
    r = dict(**meta, time=datetime.datetime.now().strftime("%Y-%m-%d %H:%M"), n=len(df),
             DSC=float(100 * df.DSC.mean()), NSD=float(100 * df.NSD.mean()),
             DSC_nonempty=float(100 * df.loc[~ge, "DSC"].mean()), NSD_nonempty=float(100 * df.loc[~ge, "NSD"].mean()),
             n_empty=int(ge.sum()), normals_pred_empty=int((ge & (pe == True)).sum()),
             lesions_missed=int((~ge & (pe == True)).sum()) if pe.notna().all() else None)
    r["paper_DSC"] = PAPER.get((meta["target"], meta["pct"])) if meta["prompt"] == "original" else None
    return r

def evaluate(target, source=None, pct=100, prompt="original", out="outputs_medclipseg", opts="", tag="", seed=SEED, allow_cpu=False):
    source = source or target
    dev = ""
    if not HAS_GPU:
        if not allow_cpu: raise RuntimeError("No GPU in this runtime: evaluation would be very slow on CPU. Switch runtime, or pass allow_cpu=True.")
        dev = "MODEL.DEVICE cpu"
    t0 = time.time()
    sh(f"python test.py --config-file configs/{target}.yaml --output-dir {out} --source_dataset {source} "
       f"--data_percentage {pct} --prompt_design {prompt} --seed {seed} {opts} {dev}")
    sh(f"python utils/eval.py --config-file configs/{target}.yaml --output-dir {out} "
       f"--data_percentage {pct} --prompt_design {prompt} --seed {seed} {opts}")
    secs = round(time.time() - t0, 1)
    name = target if pct == 100 else f"{target}_{pct}"
    seg_dir = f"{out}/{name}/seg_results/seed{seed}/{SUB.format(prompt)}"
    unc_dir = f"{out}/{name}/unc_results/seed{seed}/{SUB.format(prompt)}"
    df = enrich(pd.read_csv(f"{out}/{name}/seg_results/seed{seed}/test_Prompt-{prompt}.csv"), target, prompt, seg_dir)
    rid = run_id(target, source, pct, prompt, seed, opts, out)
    meta = dict(tag=tag, source=source, target=target, pct=pct, prompt=prompt, seed=seed, opts=opts, out=out, run_id=rid)
    for k, v in meta.items(): df[k] = v
    df.to_csv(f"{PER_IMG_DIR}/{rid}.csv", index=False)
    if SAVE_MAPS:
        with zipfile.ZipFile(f"{MAP_DIR}/{rid}.zip", "w") as z:
            for kind, d in [("mask", seg_dir), ("uncertainty", unc_dir)]:
                for f in os.listdir(d): z.write(f"{d}/{f}", f"{kind}/{f}")
    r = summarize(df, **meta); r["seconds"] = secs
    RESULTS.append(r); pd.DataFrame(RESULTS).to_csv(ALL_CSV, index=False)
    print({k: (round(v, 2) if isinstance(v, float) else v) for k, v in r.items() if k not in ("run_id", "time", "out")})
    return df

def load_per_image(target, prompt="original", source=None, pct=100, seed=SEED, opts="", out="outputs_medclipseg"):
    source = source or ("BUSI" if target in BUS else target)
    p = f"{PER_IMG_DIR}/{run_id(target, source, pct, prompt, seed, opts, out)}.csv"
    if os.path.exists(p): return pd.read_csv(p)
    name = target if pct == 100 else f"{target}_{pct}"
    p = f"{PER_IMG_DIR}/legacy__{out}__{name}__s{seed}__{prompt}.csv"
    if os.path.exists(p): return pd.read_csv(p)
    raise FileNotFoundError(f"no per-image results for {target}/{prompt} (pct={pct}, seed={seed}, out={out}) — run evaluate() first")

COLS = ["tag", "source", "target", "pct", "prompt", "seed", "opts", "n", "DSC", "NSD", "DSC_nonempty",
        "n_empty", "normals_pred_empty", "lesions_missed", "paper_DSC", "delta", "seconds"]
def show_results(tag_prefix=None):
    d = pd.DataFrame(RESULTS)
    if d.empty: print("no results yet"); return
    d["delta"] = d.DSC - pd.to_numeric(d.get("paper_DSC"), errors="coerce")
    if tag_prefix: d = d[d.tag.astype(str).str.startswith(tag_prefix)]
    display(d[[c for c in COLS if c in d]].round(2))

def save_tables():
    d = pd.DataFrame(RESULTS)
    sheets = {"all_results": d}
    if len(d):
        d = d.assign(delta=d.DSC - pd.to_numeric(d.get("paper_DSC"), errors="coerce"))
        sheets["reproduction"] = d[d.paper_DSC.notna()][[c for c in COLS if c in d]]
        p = d[d.tag.astype(str).str.match(r"T2[ad]") & (d.opts.fillna("") == "")]
        if len(p):
            for v in ["DSC", "DSC_nonempty", "normals_pred_empty"]:
                sheets[f"prompts_{v}"] = p.pivot_table(index=["source", "target"], columns="prompt", values=v, aggfunc="last")
        m = d[d.tag.astype(str).str.startswith("T2b")]
        if len(m): sheets["mc_samples"] = m[[c for c in COLS if c in m]]
    sheets.update(TABLES)
    path = f"{TAB_DIR}/medclipseg_tables.xlsx"
    with pd.ExcelWriter(path) as xw:
        for name, t in sheets.items(): t.to_excel(xw, sheet_name=name[:31])
    for name, t in sheets.items(): t.to_csv(f"{TAB_DIR}/{name}.csv")
    print(f"saved {len(sheets)} tables -> {path}")

def make_shuffled(target, seed=0):
    # every image gets another test image's original prompt (deterministic for a given seed)
    p = f"data/{target}/Prompts_Folder/"
    df = pd.read_excel(p + "Test_text_original.xlsx")
    df["Description"] = df["Description"].sample(frac=1, random_state=seed).values
    df.to_excel(p + "Test_text_shuffled.xlsx", index=False)

for t in ["BUSI", "UDIAT"]: make_shuffled(t)

## R. Restore the first session's results (run once; safe to re‑run)
Session 1 saved `results_<time>.csv` (summary table) and `csv_logs_<time>.tgz` (per‑image CSVs + training logs) in `results/`. This imports both into the new format. Per‑image files from that session can only be partly enriched: predicted masks were not kept, so `pred_empty` is known only for empty‑GT images (DSC = 1 ⇔ nothing predicted). If an evaluation ran several times (the MC‑sample sweep), only its last version survived.

In [ ]:
done = {r.get("legacy_file") for r in RESULTS}
for f in sorted(glob.glob(f"{RES_DIR}/results_*.csv")):
    if os.path.basename(f) in done: print("[skip]", os.path.basename(f)); continue
    old = pd.read_csv(f)
    t2c = iter([1, 2])
    for _, row in old.iterrows():
        r = row.to_dict()
        r["seed"] = next(t2c) if str(r["tag"]).startswith("T2c") else SEED
        r["opts"] = "" if pd.isna(r.get("opts")) else r["opts"]
        r["out"] = "outputs_train" if str(r["tag"]).startswith("T3") else "outputs_medclipseg"
        n, ne = r["n"], r["n_empty"]
        r["normals_pred_empty"] = int(round((n * r["DSC"] - (n - ne) * r["DSC_nonempty"]) / 100)) if ne else 0
        r["legacy_file"] = os.path.basename(f); r["run_id"] = "legacy"
        RESULTS.append(r)
    print("[ok]  ", os.path.basename(f), len(old), "rows")
pd.DataFrame(RESULTS).to_csv(ALL_CSV, index=False)

for tgz in sorted(glob.glob(f"{RES_DIR}/csv_logs_*.tgz")):
    dst = f"/content/restored/{os.path.basename(tgz)[:-4]}"
    shutil.rmtree(dst, ignore_errors=True); os.makedirs(dst)
    subprocess.run(["tar", "-xzf", tgz, "-C", dst], check=True)
    for p in glob.glob(f"{dst}/**/*.csv", recursive=True) + glob.glob(f"{dst}/**/log.txt", recursive=True):
        rel = os.path.relpath(p, dst).split(os.sep)             # out/name/(seg_results|trained_models)/seedN/file
        out, name, kind, seed_dir, fname = rel[0], rel[1], rel[2], rel[3], rel[-1]
        target, s = re.sub(r"_\d+$", "", name), seed_dir.replace("seed", "")
        if fname == "log.txt":
            shutil.copy(p, f"{LOG_DIR}/legacy__{out}__{name}__s{s}__{kind}.txt"); continue
        prompt = fname[len("test_Prompt-"):-4]
        if not os.path.isdir(f"data/{target}/Test_Folder/label"): print("  (no labels for", target, "- skipped)"); continue
        df = enrich(pd.read_csv(p), target, prompt)
        df.to_csv(f"{PER_IMG_DIR}/legacy__{out}__{name}__s{s}__{prompt}.csv", index=False)
    print("[ok]  ", os.path.basename(tgz))
print(len(os.listdir(PER_IMG_DIR)), "per-image files,", len(os.listdir(LOG_DIR)), "training logs on Drive")
show_results()

## A. Data understanding (CPU is fine)
### A1. Prompt inventory
Every prompt file shipped with the repo: size, number of distinct prompts, length, how many look like "normal" prompts, and the most frequent prompt. Works for all 16 datasets because the prompt files are in the repo (no image download needed).

In [ ]:
DS_ALL = ["BUSI", "BUSBRA", "BUSUC", "BUID", "UDIAT", "Kvasir", "ColonDB", "ClinicDB", "CVC300", "BKAI",
          "ISIC", "UWaterlooSkinCancer", "BTMRI", "BRISC", "Covid19", "EUS"]
NORMAL_RE = r"\bno (?:tumou?r|polyp|lesion)|\bnormal\b|unremarkable|healthy"
rows = []
for ds in DS_ALL:
    for f in sorted(glob.glob(f"data/{ds}/Prompts_Folder/*.xlsx")):
        x = pd.read_excel(f)
        if "Description" not in x or x.empty: continue          # e.g. empty Val files of ColonDB / CVC300
        s = x["Description"].astype(str).str.strip(); vc = s.value_counts()
        rows.append(dict(dataset=ds, file=os.path.basename(f)[:-5], n=len(s), unique=s.nunique(),
                         mean_words=round(s.str.split().str.len().mean(), 1),
                         normal_like=int(s.str.contains(NORMAL_RE, case=False).sum()),
                         top_prompt=vc.index[0][:90], top_prompt_n=int(vc.iloc[0])))
inv = pd.DataFrame(rows); TABLES["A1_prompt_inventory"] = inv
pd.set_option("display.max_colwidth", 90)
display(inv[inv.dataset.isin(["BUSI", "UDIAT"])])

In [ ]:
# The same BUSI test image under every prompt design (one lesion image, one normal image)
designs = ["original", "missing_location", "underdescriptive", "overdescriptive", "contradictory", "shuffled"]
P = {d: prompts_for("BUSI", d) for d in designs}
o = P["original"]
for name in [o[~o.str.startswith("No tumor")].index[3], o[o.str.startswith("No tumor")].index[0]]:
    print(f"\n{name}"); [print(f"  {d:17s} {P[d][name]}") for d in designs]

### A2. Are the BUSI prompt "designs" controlled edits of the original?
The paper (Table 4) reads e.g. `missing_location` as "the original prompt without the location". This parses size, shape, class and location from each BUSI prompt and counts, per design, how often an attribute **changed** (both prompts state it, values differ) or was **dropped**. Only BUSI uses this template; the OOD sets use a different sentence style.

In [ ]:
def attrs(s):
    s = s.lower()
    return dict(size=tuple(re.findall(r"\b(small|medium|large)\b", s)),
                shape=tuple(re.findall(r"\b([\w]+)-shaped\b", s)),
                cls=tuple(re.findall(r"\b(benign|malignant|regular|irregular)\b", s)),
                loc=tuple(re.findall(r"\b(top left|top right|bottom left|bottom right|top|bottom|left|right|center)\b", s)))
rows = []
A0 = {k: attrs(v) for k, v in o.items()}
lesion = [k for k, a in A0.items() if a["shape"]]
for d in designs[1:-1]:
    Ad = {k: attrs(v) for k, v in P[d].items()}
    r = dict(design=d, lesion_prompts=len(lesion))
    for a in ["size", "shape", "cls", "loc"]:
        r[f"{a}_changed"] = sum(1 for k in lesion if Ad[k][a] and A0[k][a] and Ad[k][a] != A0[k][a])
        r[f"{a}_dropped"] = sum(1 for k in lesion if A0[k][a] and not Ad[k][a])
    rows.append(r)
TABLES["A2_busi_design_consistency"] = pd.DataFrame(rows)
display(TABLES["A2_busi_design_consistency"])

### A3. Do the prompts describe the ground‑truth mask?
For every image with a non‑empty mask: the stated location vs. the centroid of the largest mask component (3×3 grid; an axis the prompt doesn't mention counts as middle/center), stated lesion count vs. connected components, stated size vs. mask area. Also normal‑prompt ↔ empty‑mask agreement. High agreement means the text carries information taken from the annotation.

In [ ]:
from scipy import ndimage
NUM = {"one": 1, "two": 2, "three": 3, "four": 4, "five": 5}
def stated_loc(s):
    s = s.lower().replace("ultrasound image", "")
    v = re.search(r"\b(top|upper|bottom|lower)\b", s); h = re.search(r"\b(left|right)\b", s)
    c = re.search(r"\b(cent(?:er|re|ral)|middle)\b", s)
    if not (v or h or c): return None
    return ("top" if v and v.group(1) in ("top", "upper") else "bottom" if v else "middle", h.group(1) if h else "center")
def mask_cell(m):
    lab, n = ndimage.label(m > 127)
    if n == 0: return None, 0
    sizes = ndimage.sum(np.ones_like(lab), lab, range(1, n + 1))
    big = int(np.argmax(sizes)) + 1
    cy, cx = ndimage.center_of_mass(lab == big)
    H, W = m.shape
    v = "top" if cy < H / 3 else "bottom" if cy > 2 * H / 3 else "middle"
    h = "left" if cx < W / 3 else "right" if cx > 2 * W / 3 else "center"
    n_big = int((sizes >= 0.001 * m.size).sum())            # ignore specks < 0.1% of the image
    return (v, h), n_big

rows, per = [], []
for ds in BUS:
    for split, xl in [("Train", "Train_text.xlsx"), ("Val", "Val_text.xlsx"), ("Test", "Test_text_original.xlsx")]:
        p = f"data/{ds}/Prompts_Folder/{xl}"
        if not (os.path.exists(p) and os.path.isdir(f"data/{ds}/{split}_Folder/label")): continue
        df = pd.read_excel(p)
        for _, r in df.iterrows():
            m = cv2.imread(f"data/{ds}/{split}_Folder/label/{r['Ground Truth']}", 0)
            if m is None: continue
            s = str(r["Description"]).strip(); cell, ncomp = mask_cell(m)
            cnt = re.match(r"^(one|two|three|four|five)\b", s.lower())
            size = re.search(r"\b(small|medium|large)\b", s.lower())
            per.append(dict(dataset=ds, split=split, mask_empty=cell is None, normal_prompt=bool(re.search(NORMAL_RE, s, re.I)),
                            stated=stated_loc(s), mask_cell=cell, stated_count=NUM[cnt.group(1)] if cnt else None,
                            components=ncomp, stated_size=size.group(1) if size else None, area=float((m > 127).mean())))
per = pd.DataFrame(per)
for (ds, split), g in per.groupby(["dataset", "split"], sort=False):
    les = g[~g.mask_empty]; st = les[les.stated.notna()]
    one = st[(st.stated_count.isna()) | (st.stated_count == 1)]
    rows.append(dict(dataset=ds, split=split, images=len(g), empty_masks=int(g.mask_empty.sum()),
                     normal_prompt_and_empty=int((g.mask_empty & g.normal_prompt).sum()),
                     normal_prompt_but_lesion=int((~g.mask_empty & g.normal_prompt).sum()),
                     empty_mask_but_lesion_prompt=int((g.mask_empty & ~g.normal_prompt).sum()),
                     lesions_with_location=len(st),
                     loc_exact_3x3=round(100 * (one.stated == one.mask_cell).mean(), 1) if len(one) else None,
                     loc_vertical=round(100 * (one.stated.str[0] == one.mask_cell.str[0]).mean(), 1) if len(one) else None,
                     loc_horizontal=round(100 * (one.stated.str[1] == one.mask_cell.str[1]).mean(), 1) if len(one) else None,
                     count_matches=round(100 * (les.stated_count == les.components)[les.stated_count.notna()].mean(), 1) if les.stated_count.notna().any() else None))
TABLES["A3_prompt_vs_mask"] = pd.DataFrame(rows)
display(TABLES["A3_prompt_vs_mask"])
b = per[(per.dataset == "BUSI") & ~per.mask_empty & per.stated_size.notna()]
TABLES["A3_busi_size_vs_area"] = b.groupby("stated_size").area.describe()[["count", "25%", "50%", "75%"]].mul([1, 100, 100, 100]).round(2)
print("BUSI: mask area (% of image) by stated size"); display(TABLES["A3_busi_size_vs_area"])
x = per[(per.dataset == "BUSI") & ~per.mask_empty & per.stated.notna()]
TABLES["A3_busi_location_crosstab"] = pd.crosstab(x.stated.astype(str), x.mask_cell.astype(str))
print("BUSI: stated location (rows) vs mask-centroid cell (cols)"); display(TABLES["A3_busi_location_crosstab"])
save_tables()

### A4. Look at a few BUSI test images with their mask and prompt

In [ ]:
import matplotlib.pyplot as plt, textwrap
df = pd.read_excel("data/BUSI/Prompts_Folder/Test_text_original.xlsx").sample(8, random_state=1)
fig, ax = plt.subplots(2, 4, figsize=(16, 8.5))
for a, (_, r) in zip(ax.ravel(), df.iterrows()):
    img = cv2.cvtColor(cv2.imread(f"data/BUSI/Test_Folder/img/{r['Image']}"), cv2.COLOR_BGR2RGB)
    m = cv2.imread(f"data/BUSI/Test_Folder/label/{r['Ground Truth']}", 0)
    cs, _ = cv2.findContours((m > 127).astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
    cv2.drawContours(img, cs, -1, (255, 0, 0), 2)
    a.imshow(img); a.set_title("\n".join(textwrap.wrap(r["Description"], 45)), fontsize=8); a.axis("off")
plt.tight_layout(); plt.show()

## B. Claim checks (CPU is fine; B3 is slow‑ish on CPU)
### B1. Normal BUSI test images under each prompt design
Uses the per‑image results (from Section R or from new runs). For an empty ground truth the DSC is binary: **1 = the model predicted nothing**, 0 = it predicted a lesion that isn't there.

In [ ]:
tabs = {}
for d in designs:
    try: tabs[d] = load_per_image("BUSI", d).set_index("Name")
    except FileNotFoundError as e: print("missing:", d)
normals = next(iter(tabs.values())).query("gt_empty").index
B1 = pd.DataFrame({d: t.loc[normals, "DSC"] for d, t in tabs.items()})
if "shuffled" in tabs:
    B1["prompt_given_when_shuffled"] = tabs["shuffled"].loc[normals, "prompt_text"]
    B1["shuffled_prompt_says_normal"] = B1.prompt_given_when_shuffled.str.contains(NORMAL_RE, case=False)
TABLES["B1_busi_normals_by_design"] = B1.reset_index()
display(B1)
print("normal images predicted empty, per design:", B1[[d for d in designs if d in B1]].sum().astype(int).to_dict())
if "shuffled" in B1:
    print(B1.groupby("shuffled_prompt_says_normal")["shuffled"].agg(["count", "sum"]).rename(columns={"sum": "predicted_empty"}))
save_tables()

### B2. The Dice loss in `train.py` is applied to `(B, H, W)` tensors
`calc_loss` passes logits and masks of shape `(B, 224, 224)` to MONAI's `DiceLoss`, which expects `(B, C, H, W)`. MONAI therefore treats the 224 image **rows as 224 classes**, drops row 0 (`include_background=False`) and computes a Dice per row. A row with no lesion only gets a low loss when the sum of its predicted probabilities is ≪ 1e‑5, i.e. background logits below about −17. Synthetic check with a **perfect** mask (no model needed):

In [ ]:
from monai.losses import DiceLoss
dl = DiceLoss(include_background=False, sigmoid=True, reduction="mean")      # exactly as in train.py
gt = torch.zeros(2, 224, 224); gt[0, 80:140, 90:150] = 1                       # image 0: lesion, image 1: normal
rows = []
for bg in [-2, -5, -10, -15, -17, -20, -25]:
    lg = torch.where(gt.bool(), torch.tensor(5.0), torch.tensor(float(bg)))   # perfect segmentation
    rows.append(dict(background_logit=bg, repo_loss_BHW=round(dl(lg, gt).item(), 3),
                     proper_loss_B1HW=round(dl(lg.unsqueeze(1), gt.unsqueeze(1)).item(), 3)))
TABLES["B2_dice_shape_demo"] = pd.DataFrame(rows); display(TABLES["B2_dice_shape_demo"])

### B3. Diagnostics on the released BUSI checkpoint
1. **Logits:** are background logits pushed below the −17 "cliff" the row‑wise Dice rewards? (Would explain the sudden drop of train *and* val loss around epoch 11 while val Dice rose smoothly.)
2. **Loss components** on a validation batch: repo Dice (row‑wise), proper Dice, BCE.
3. **Soft contrastive targets:** the code divides the text‑similarity logits by their row L2 norm before the softmax; paper Eq. 20 doesn't. How flat are the targets in each version? (1.0 normalised entropy = uniform.)

On CPU this takes a few minutes (model load + two batches).

In [ ]:
sys.path.insert(0, REPO)
from utils.main_utils import load_cfg_from_cfg_file, read_text
from trainers import build_medclipseg_unimedclip
from datasets.dataloader import DatasetSegmentation, ValGenerator
from monai.losses import DiceLoss
dev = "cuda" if HAS_GPU else "cpu"
sh(f"python utils/download_ckpt.py --dataset BUSI --seed {SEED}")
cfg = load_cfg_from_cfg_file("configs/BUSI.yaml"); cfg.MODEL.DEVICE = dev
model = build_medclipseg_unimedclip(cfg)
ck = torch.load(f"outputs_medclipseg/BUSI/trained_models/seed{SEED}/MedCLIPSeg_unimedclip_ViT-B-16_latest.pth", map_location=dev, weights_only=False)
model.load_state_dict(ck["model"]); model.to(dev).eval()

def batch(split, xl, n, n_normal=None, seed=0):
    ds = DatasetSegmentation(f"data/BUSI/{split}_Folder/", "BUSI", read_text(f"data/BUSI/Prompts_Folder/{xl}"), ValGenerator([224, 224]), image_size=224)
    rng = np.random.default_rng(seed); idx = rng.permutation(len(ds))
    if n_normal is not None:
        nor = [i for i in idx if ds.data_pairs[i][2].startswith("No tumor")][:n_normal]
        idx = nor + [i for i in idx if not ds.data_pairs[i][2].startswith("No tumor")][:n - n_normal]
    items = [ds[int(i)] for i in idx[:n]]
    return (torch.stack([b["image"] for b in items]).to(dev), torch.stack([b["ground_truth_mask"] for b in items]).to(dev),
            [b["text_prompt"] for b in items])

imgs, masks, texts = batch("Val", "Val_text.xlsx", 12, n_normal=4)
with torch.no_grad():
    logits = model(image=imgs, text=texts, num_samples=1)[0]
m = masks.bool(); normal_img = ~m.flatten(1).any(1)
q = lambda x: np.percentile(x.float().cpu().numpy(), [1, 50, 99]).round(1).tolist()
B3a = pd.DataFrame([
    dict(pixels="background, lesion images", n=int((~m[~normal_img]).sum()), logit_p1_p50_p99=q(logits[~normal_img][~m[~normal_img]]), frac_below_minus17=float((logits[~normal_img][~m[~normal_img]] < -17).float().mean())),
    dict(pixels="background, normal images", n=int((~m[normal_img]).sum()), logit_p1_p50_p99=q(logits[normal_img]), frac_below_minus17=float((logits[normal_img] < -17).float().mean())),
    dict(pixels="lesion", n=int(m.sum()), logit_p1_p50_p99=q(logits[m]), frac_below_minus17=float((logits[m] < -17).float().mean())),
])
dl = DiceLoss(include_background=False, sigmoid=True); bce = torch.nn.BCEWithLogitsLoss()
B3b = pd.DataFrame([dict(model_logits=lab, repo_dice_BHW=dl(L, masks).item(), proper_dice_B1HW=DiceLoss(sigmoid=True)(L.unsqueeze(1), masks.unsqueeze(1)).item(),
                         bce=bce(L, masks.float()).item(), val_dice_at_0_5=(2 * ((L > 0) & m).sum() / ((L > 0).sum() + m.sum())).item())
                    for lab, L in [("as predicted", logits), ("same, logits clamped to >= -10", logits.clamp(min=-10))]]).round(3)
TABLES["B3a_logit_stats"], TABLES["B3b_loss_components"] = B3a, B3b
display(B3a); display(B3b)

In [ ]:
# Soft contrastive targets on a training-size batch (24) of training images + prompts
imgs, _, texts = batch("Train", "Train_text.xlsx", 24, seed=1)
with torch.no_grad():
    tok = model.tokenizer(texts).to(dev)
    emb = model.text_model.transformer.embeddings.word_embeddings(tok).type(model.dtype)
    _, txt = model.encode_text_image(tok, emb, imgs)
T = cfg.MODEL.TEMPERATURE
sim = txt @ txt.T / T
G_code = torch.softmax(sim / sim.norm(dim=-1, keepdim=True), -1)          # as in trainers/medclipseg_unimedclip.py
tn = txt / txt.norm(dim=-1, keepdim=True)
G_paper = torch.softmax(tn @ tn.T / T, -1)                                   # Eq. 20 with L2-normalised text features
def flat(G):
    H = -(G * G.clamp_min(1e-12).log()).sum(-1) / np.log(G.shape[1])
    return dict(max_prob=round(G.max(-1).values.mean().item(), 3), own_pair_prob=round(G.diag().mean().item(), 3),
                normalised_entropy=round(H.mean().item(), 3))
B3c = pd.DataFrame([dict(version="uniform (1/24)", max_prob=round(1/24, 3), own_pair_prob=round(1/24, 3), normalised_entropy=1.0),
                    dict(version="code (row-L2-normalised logits)", **flat(G_code)),
                    dict(version="paper Eq. 20 (normalised text)", **flat(G_paper))])
print("distinct prompts in this batch:", len(set(texts)), "of", len(texts))
TABLES["B3c_soft_targets"] = B3c; display(B3c)
save_tables()
del model; import gc; gc.collect()

## T1. Reproduce the paper from the released checkpoints (GPU)
`utils/download_ckpt.py` puts the authors' checkpoint where `test.py` looks. **Pass:** each DSC within ~±1 of the paper. (Already done in session 1 — Section R restored those rows; re‑run only if needed.)

In [ ]:
sh(f"python utils/download_ckpt.py --dataset BUSI --seed {SEED}")
evaluate("BUSI", tag="T1-ID")                                     # paper: 85.72
for tgt in ["BUSBRA", "BUSUC", "BUID", "UDIAT"]:                  # paper: 75.06 / 84.37 / 78.99 / 74.64
    evaluate(tgt, source="BUSI", tag="T1-OOD")
for pct in [10, 25, 50]:                                          # paper: 68.66 / 77.73 / 81.48
    sh(f"python utils/download_ckpt.py --dataset BUSI --data_percentage {pct} --seed {SEED}")
    evaluate("BUSI", pct=pct, tag="T1-eff")
show_results("T1")

## T2. Probes on the released checkpoint (GPU)
### T2a. Prompt designs shipped with the repo + shuffled prompts

In [ ]:
for tgt in ["BUSI", "UDIAT"]:
    for design in designs:
        evaluate(tgt, source="BUSI", prompt=design, tag="T2a-prompt")
show_results("T2a")

### T2d. Controlled prompt edits (one attribute at a time) — BUSI test set
Because the repo's designs change several attributes at once (Section A2), these edits change exactly one thing in each lesion prompt and leave normal prompts untouched — except the last two, which test whether the model obeys the normal/lesion statement:

| edit | lesion prompts | normal prompts |
|---|---|---|
| `no_location` / `flip_location` | location removed / top↔bottom, left↔right | unchanged |
| `no_shape`, `no_size`, `no_class` | that attribute removed | unchanged |
| `generic` | "One tumor in the breast ultrasound image." (count kept) | unchanged |
| `lesion_as_normal` | "No tumor in the breast ultrasound image." | unchanged |
| `normal_as_lesion` | unchanged | "One tumor in the breast ultrasound image." |

In [ ]:
PAT = re.compile(r"^(One|Two|Three|Four|Five) (small|medium|large) ([\w-]+?)-shaped (benign|malignant|regular|irregular) (tumors?) at the (.+?) in the breast ultrasound image\.$")
NORMAL_TXT = "No tumor in the breast ultrasound image."
FLIP = {"top": "bottom", "bottom": "top", "left": "right", "right": "left"}
def edit(s, kind):
    s = str(s).strip()
    if s == NORMAL_TXT:
        return "One tumor in the breast ultrasound image." if kind == "normal_as_lesion" else s
    m = PAT.match(s)
    if not m: return s
    cnt, size, shape, cls, noun, loc = m.groups()
    if kind == "generic": return f"{cnt} {noun} in the breast ultrasound image."
    if kind == "lesion_as_normal": return NORMAL_TXT
    parts = [cnt, size, f"{shape}-shaped", cls, noun, f"at the {loc}"]
    if kind == "no_location": parts[5] = ""
    if kind == "flip_location": parts[5] = "at the " + re.sub(r"\b(top|bottom|left|right)\b", lambda w: FLIP[w.group(1)], loc)
    if kind == "no_shape": parts[2] = ""
    if kind == "no_size": parts[1] = ""
    if kind == "no_class": parts[3] = ""
    return " ".join(p for p in parts if p) + " in the breast ultrasound image."

base = pd.read_excel("data/BUSI/Prompts_Folder/Test_text_original.xlsx")
assert all(edit(s, "rebuild") == str(s).strip() for s in base.Description), "template parser does not reproduce the original prompts"
EDITS = ["no_location", "flip_location", "no_shape", "no_size", "no_class", "generic", "lesion_as_normal", "normal_as_lesion"]
for k in EDITS:
    e = base.copy(); e["Description"] = e.Description.map(lambda s: edit(s, k))
    e.to_excel(f"data/BUSI/Prompts_Folder/Test_text_{k}.xlsx", index=False)
print(base.Description.iloc[0]); [print(f"  {k:17s}", edit(base.Description.iloc[0], k)) for k in EDITS]

In [ ]:
for k in EDITS:
    evaluate("BUSI", prompt=k, tag="T2d-edit")
show_results("T2d")
save_tables()

### T2e. Follow-up prompt edits — size sweep, negation, which word restores detection (BUSI test, GPU ~18 min)
T2d showed that the size word sets mask extent and that "One tumor…" behaves like "No tumor…". These edits test that directly:

| edit | lesion prompts | normal prompts | question |
|---|---|---|---|
| `size_small` / `size_medium` / `size_large` | size word set to that value | unchanged | does the size word control mask extent? |
| `negated_full` | "No small oval-shaped … at the …" | unchanged | does the model read "No" in a full prompt? |
| `size_only` / `location_only` | "One small tumor in …" / "One tumor at the top in …" | unchanged | which word restores detection vs `generic`? |
| `normal_as_full_lesion` | unchanged | "One medium oval-shaped benign tumor at the center in …" | does a detailed prompt create a lesion? |

T2e‑3 needs the T2d per-image results (`original`, `no_size`, `generic`, `lesion_as_normal`, `normal_as_lesion`) on Drive; it reports anything missing.

In [ ]:
# ==== T2e-1. Follow-up prompt edits: build the prompt files (BUSI test set) ====
# Needs: Section 0 + the helpers cell (evaluate, load_per_image, show_results, save_tables).
import re
BUS = globals().get("BUS", ["BUSI", "BUSBRA", "BUSUC", "BUID", "UDIAT"])   # load_per_image() needs it
P2E = "data/BUSI/Prompts_Folder/"
PAT = re.compile(r"^(One|Two|Three|Four|Five) (small|medium|large) ([\w-]+?)-shaped "
                 r"(benign|malignant|regular|irregular) (tumors?) at the (.+?) in the breast ultrasound image\.$")
NORMAL_TXT = "No tumor in the breast ultrasound image."
FULL_LESION_TXT = "One medium oval-shaped benign tumor at the center in the breast ultrasound image."
END = " in the breast ultrasound image."

def edit2(s, kind):
    s = str(s).strip()
    if s == NORMAL_TXT:                                   # normal images: only one edit touches them
        return FULL_LESION_TXT if kind == "normal_as_full_lesion" else s
    m = PAT.match(s)
    if m is None:                                         # fail loudly instead of silently keeping the prompt
        raise ValueError(f"unparsed prompt: {s!r}")
    cnt, size, shape, cls, noun, loc = m.groups()
    if kind.startswith("size_") and kind != "size_only":
        size = kind[len("size_"):]                        # size_small / size_medium / size_large
    elif kind == "negated_full":  cnt = "No"              # "No small oval-shaped ... tumor at ..."
    elif kind == "size_only":     return f"{cnt} {size} {noun}{END}"
    elif kind == "location_only": return f"{cnt} {noun} at the {loc}{END}"
    elif kind not in ("rebuild", "normal_as_full_lesion"):
        raise ValueError(kind)
    return f"{cnt} {size} {shape}-shaped {cls} {noun} at the {loc}{END}"

EDITS2 = ["size_small", "size_medium", "size_large",          # causal test: does the size word set mask extent?
          "negated_full",                                     # negation inside the full template
          "size_only", "location_only",                       # which attribute restores detection vs. generic
          "normal_as_full_lesion"]                            # normals get a detailed lesion prompt (controlled B1)

base2 = pd.read_excel(P2E + "Test_text_original.xlsx")
assert all(edit2(s, "rebuild") == str(s).strip() for s in base2.Description), "parser does not reproduce the originals"
for k in EDITS2:
    e = base2.copy(); e["Description"] = e.Description.map(lambda s: edit2(s, k))
    n_changed = (e.Description != base2.Description.str.strip()).sum()
    e.to_excel(P2E + f"Test_text_{k}.xlsx", index=False)
    print(f"{k:22s} {n_changed:2d}/78 prompts changed | e.g. {e.Description.iloc[0] if k != 'normal_as_full_lesion' else e.Description.iloc[1]}")

In [ ]:
# ==== T2e-2. Run them (GPU, ~2.5 min each, ~18 min total) ====
for k in EDITS2:
    evaluate("BUSI", prompt=k, tag="T2e-edit")
show_results("T2e")
save_tables()

In [ ]:
# ==== T2e-3. Analysis (CPU is fine; reads the per-image CSVs on Drive) ====
def _pi(prompt):
    try: return load_per_image("BUSI", prompt).set_index("Name")
    except FileNotFoundError: print(f"  (missing per-image results for '{prompt}' - run evaluate() for it)"); return None

orig = _pi("original")
lesions = orig.index[~orig.gt_empty.astype(bool)]
normals = orig.index[orig.gt_empty.astype(bool)]
stated = orig.loc[lesions, "prompt_text"].astype(str).str.extract(r"^\w+ (small|medium|large) ")[0]
SIZES = ["small", "medium", "large"]

# (a) Size sweep: same image, only the size word changes -------------------------------------------
size_conds = {"original": "original", "none": "no_size", "small": "size_small", "medium": "size_medium", "large": "size_large"}
st = {lab: t for lab, p in size_conds.items() if (t := _pi(p)) is not None}
ratio = pd.DataFrame({lab: t.loc[lesions, "pred_area"] / t.loc[lesions, "gt_area"] for lab, t in st.items()})
dsc = pd.DataFrame({lab: 100 * t.loc[lesions, "DSC"] for lab, t in st.items()})
ratio["true size"] = dsc["true size"] = stated
T2e_ratio = ratio.groupby("true size").median().reindex(SIZES).round(2)
T2e_ratio.insert(0, "n", stated.value_counts().reindex(SIZES))
T2e_dsc = dsc.groupby("true size").mean().reindex(SIZES).round(1)
print("Median predicted/true lesion area - rows: size word in the ORIGINAL prompt, cols: size word GIVEN"); display(T2e_ratio)
print("Mean lesion DSC (%)"); display(T2e_dsc)
if all(s in st for s in SIZES):
    a = pd.DataFrame({s: st[s].loc[lesions, "pred_area"] for s in SIZES})
    mono = ((a.small < a.medium) & (a.medium < a.large)).mean()
    print(f"Lesions whose predicted area grows small < medium < large: {100 * mono:.0f}%  "
          f"(median area ×{(a.large / a.small.clip(lower=1e-6)).median():.2f} from 'small' to 'large')")
    fig, ax = plt.subplots(figsize=(5, 3.5))
    for s in SIZES:
        ax.plot(SIZES, T2e_ratio.loc[s, SIZES], marker="o", label=f"true size: {s}")
    ax.axhline(1, color="gray", lw=0.8, ls="--"); ax.set_yscale("log")
    ax.set_xlabel("size word given in the prompt"); ax.set_ylabel("median predicted / true area")
    ax.set_title("Does the size word set the mask extent?", fontsize=10); ax.legend(fontsize=8); plt.tight_layout(); plt.show()

# (b) Presence: which wording makes lesions disappear? --------------------------------------------
pres_conds = ["original", "negated_full", "location_only", "size_only", "generic", "lesion_as_normal"]
pt = {p: t for p in pres_conds if (t := _pi(p)) is not None}
missed = {p: set(t.loc[lesions].query("pred_empty == True").index) for p, t in pt.items()}
T2e_presence = pd.DataFrame({p: {"example prompt": t.loc[lesions[0], "prompt_text"],
                                 "lesion DSC": round(100 * t.loc[lesions, "DSC"].mean(), 1),
                                 "lesions missed": len(missed[p]),
                                 "median pred/true area": round((t.loc[lesions, "pred_area"] / t.loc[lesions, "gt_area"]).median(), 2)}
                             for p, t in pt.items()}).T
print("Lesion images (65) under presence-related edits"); display(T2e_presence)
T2e_overlap = pd.DataFrame({a: {b: len(missed[a] & missed[b]) for b in missed} for a in missed})
print("Missed-lesion overlap (cell = number of lesions missed under BOTH; diagonal = missed under that edit)"); display(T2e_overlap)

# (c) Normals: does a detailed lesion prompt create a lesion? ---------------------------------------
norm_conds = ["original", "normal_as_lesion", "normal_as_full_lesion"]
nt = {p: t for p in norm_conds if (t := _pi(p)) is not None}
T2e_normals = pd.DataFrame({p: {"prompt given to normals": t.loc[normals[0], "prompt_text"],
                                "normals with a false positive": int((t.loc[normals, "pred_empty"] != True).sum()),
                                "mean predicted area (% image)": round(100 * t.loc[normals, "pred_area"].mean(), 2)}
                            for p, t in nt.items()}).T
print("Normal images (13)"); display(T2e_normals)

for name, t in {"T2e_size_area_ratio": T2e_ratio, "T2e_size_dsc": T2e_dsc, "T2e_presence": T2e_presence,
                "T2e_missed_overlap": T2e_overlap, "T2e_normals": T2e_normals}.items():
    TABLES[name] = t
save_tables()

### T2b. Monte‑Carlo samples vs. accuracy and runtime

In [ ]:
for k in [1, 5, 10, 30]:
    evaluate("BUSI", opts=f"TEST.NUM_SAMPLES {k}", tag=f"T2b-mc{k}")
show_results("T2b")

### T2c. Test‑time sampling noise (same checkpoint, different `--seed`)

In [ ]:
src = f"outputs_medclipseg/BUSI/trained_models/seed{SEED}"
for s in [1, 2]:
    os.makedirs(f"outputs_medclipseg/BUSI/trained_models/seed{s}", exist_ok=True)
    shutil.copy(f"{src}/MedCLIPSeg_unimedclip_ViT-B-16_latest.pth", f"outputs_medclipseg/BUSI/trained_models/seed{s}/")
    evaluate("BUSI", seed=s, tag="T2c-mcnoise")
show_results("T2c")

### Look at predictions
`show_cases()` shows the worst and best images of any evaluation (read from the maps zip on Drive, so it also works for earlier sessions when `SAVE_MAPS` was on).

In [ ]:
def show_cases(target, prompt="original", source=None, pct=100, seed=SEED, opts="", out="outputs_medclipseg", n=4):
    source = source or ("BUSI" if target in BUS else target)
    rid = run_id(target, source, pct, prompt, seed, opts, out)
    if not (os.path.exists(f"{PER_IMG_DIR}/{rid}.csv") and os.path.exists(f"{MAP_DIR}/{rid}.zip")):
        print(f"no saved masks for run {rid} - run evaluate() for it in this or a later session (with SAVE_MAPS=True)"); return
    df = pd.read_csv(f"{PER_IMG_DIR}/{rid}.csv").sort_values("DSC")
    imgs = pd.read_excel(f"data/{target}/Prompts_Folder/Test_text_{prompt}.xlsx").set_index("Ground Truth")["Image"]
    z = zipfile.ZipFile(f"{MAP_DIR}/{rid}.zip")
    rd = lambda p, flag: cv2.imdecode(np.frombuffer(z.read(p), np.uint8), flag)
    picks = list(df.head(n // 2).Name) + list(df.tail(n - n // 2).Name)
    fig, ax = plt.subplots(len(picks), 4, figsize=(13, 3.2 * len(picks)))
    for r, m in enumerate(picks):
        row = df.set_index("Name").loc[m]
        img = cv2.cvtColor(cv2.imread(f"data/{target}/Test_Folder/img/{imgs[m]}"), cv2.COLOR_BGR2RGB)
        panels = [(img, textwrap.shorten(str(row.prompt_text), 60)), (cv2.imread(f"data/{target}/Test_Folder/label/{m}", 0), "ground truth"),
                  (rd(f"mask/{m}", 0), f"prediction  DSC={row.DSC:.2f}"), (cv2.cvtColor(rd(f"uncertainty/{m}", 1), cv2.COLOR_BGR2RGB), "uncertainty (per-image scaled)")]
        for c, (im, t) in enumerate(panels):
            ax[r, c].imshow(im, cmap="gray"); ax[r, c].set_title(t, fontsize=8); ax[r, c].axis("off")
    plt.tight_layout(); plt.show()

show_cases("BUSI")

## T3. Train from scratch (GPU)
Paper settings: batch 24, Adam 3e‑4, cosine schedule, 100 epochs, last‑epoch checkpoint. `train()` copies the training log to Drive (and the checkpoint if `KEEP_CKPT`). If Colab disconnects, re‑run setup and call `train(..., resume=True)`.

In [ ]:
OUT = "outputs_train"
def train(ds, pct=100, out=OUT, seed=SEED, opts="", resume=False, allow_cpu=False):
    if not HAS_GPU and not allow_cpu: raise RuntimeError("No GPU in this runtime: training needs a GPU.")
    dev = "" if HAS_GPU else "MODEL.DEVICE cpu"
    t0 = time.time()
    get_ipython().system(f"python train.py --config-file configs/{ds}.yaml --output-dir {out} --data_percentage {pct} --seed {seed} {'--resume' if resume else ''} {opts} {dev}")
    name = ds if pct == 100 else f"{ds}_{pct}"
    d = f"{out}/{name}/trained_models/seed{seed}"
    tag = f"{slug(out.replace('outputs_', ''))}__{name}__s{seed}__{slug(opts)}"
    if os.path.exists(f"{d}/log.txt"): shutil.copy(f"{d}/log.txt", f"{LOG_DIR}/{tag}.txt")
    ck = f"{d}/MedCLIPSeg_unimedclip_ViT-B-16_latest.pth"
    assert os.path.exists(ck) and os.path.getmtime(ck) > t0, "training failed (no new checkpoint) - see the output above"
    if KEEP_CKPT: shutil.copy(ck, f"{RES_DIR}/{tag}_latest.pth")
    print(f"training took {(time.time() - t0) / 60:.1f} min; log saved as train_logs/{tag}.txt")

def plot_log(path):
    log = open(path).read()
    ep = [(int(a), float(b), float(c)) for a, b, c in re.findall(r"EPOCH: (\d+) \| Training Loss: ([\d.]+) \| Validation Loss: ([\d.]+)", log)]
    dice = [float(x) for x in re.findall(r"(?:Dice: |→ )([\d.]+)\n", log)]
    e, tr, va = zip(*ep)
    plt.figure(figsize=(10, 3)); plt.subplot(1, 2, 1); plt.plot(e, tr, label="train"); plt.plot(e, va, label="val"); plt.legend(); plt.title("loss")
    plt.subplot(1, 2, 2); plt.plot(range(1, len(dice) + 1), dice); plt.title("val Dice (1 MC sample)"); plt.suptitle(os.path.basename(path), fontsize=8); plt.show()

print("\n".join(sorted(os.listdir(LOG_DIR))))

In [ ]:
train("BUSI", pct=10)
evaluate("BUSI", pct=10, out=OUT, tag="T3-train")
train("BUSI")
evaluate("BUSI", out=OUT, tag="T3-train")
for tgt in ["BUSBRA", "BUSUC", "BUID", "UDIAT"]:
    evaluate(tgt, source="BUSI", out=OUT, tag="T3-train")
show_results("T3")
plot_log(f"{LOG_DIR}/train__BUSI__s{SEED}__default.txt")
save_tables()

## Save everything
`save_tables()` is also called at the end of each section. Files: `results/tables/medclipseg_tables.xlsx` (one sheet per table) and one CSV per table.

In [ ]:
save_tables()
!ls -la "{TAB_DIR}"

## Next: other dataset groups (GPU)
```python
for ds in ["Kvasir", "ColonDB", "ClinicDB", "CVC300", "BKAI"]: get_dataset(ds)
sh(f"python utils/download_ckpt.py --dataset Kvasir --seed {SEED}")
evaluate("Kvasir", tag="ENDO-ID")
for t in ["ColonDB", "ClinicDB", "CVC300", "BKAI"]: evaluate(t, source="Kvasir", tag="ENDO-OOD")
```
ISIC → UWaterlooSkinCancer and BTMRI → BRISC work the same way. BTMRI's test set has many normal scans with empty masks: watch `DSC_nonempty` and `normals_pred_empty`.